In [1]:
# Testing objective function output

import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
eq_init = desc.io.load("desc_eq_new_QH_aScaling.h5")
obj = TrappedResonance(eq_init)
# grid is set in init of TrappedResonance class
obj.build()
value = obj.compute(eq_init.params_dict)
# value = eq_init.compute('f_tr',grid=grid)

print(value)
# print(np.shape(value['alpha_drift_avg']))


Precomputing transforms
[0.         1.         0.5        2.         3.14159265]


In [ ]:
print(value.shape)

In [ ]:
# debug cell (restart kernel and run before running optimizer)
import sys
import os
import math

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    AspectRatio,
    ForceBalance,
    QuasisymmetryBoozer,
    QuasisymmetryTwoTerm,
    QuasisymmetryTripleProduct,
    ObjectiveFromUser,
    GammaC,
    TrappedResonance
)
from desc.optimize import Optimizer
from desc.plotting import (
    plot_grid,
    plot_boozer_modes,
    plot_boozer_surface,
    plot_qs_error,
    plot_boundaries,
    plot_boundary,
)
# load initial equilibrium
# eq_init = desc.io.load("qs_initial_guess.h5") # QI
eq_init = desc.io.load("desc_eq_new_QH_aScaling.h5") # QH

# Specify equilibrium nfp and helicity
N = -1 # QH helicity
# N = 0 # QA
nfp = eq_init.NFP
s_input = np.linspace(0.1,0.9,3) # surfaces to test, rho = sqrt(s) is calculated below. DESC 

# optimizer = Optimizer("proximal-scipy-bfgs")
optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)

eq_qs_T = eq_init.copy()  # make a copy of the original one
# constraints
constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixIota(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# Create a grid in (rho, theta, zeta) coordinates
# rho = np.linspace(0.1,0.9,5)
rho=abs(np.sqrt(s_input))
eq_periodicity = (np.inf,np.inf,np.inf) # periodicity in zeta for these equilibrium to make rtz grid
grid = eq_init._get_rtz_grid( # returns rho, theta, zeta coordinate grid
    rho, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.linspace(0, 12 * np.pi, 300), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
# grid has the number of nodes equal to len(rho)*len(alpha)*length(zeta)

# Grid for QuasisymmetryTripleProduct objective
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)

# Objective for resonance
objective_fT = ObjectiveFunction(
    (
        QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol),
        TrappedResonance(eq=eq_qs_T, grid=grid)
    ), 
)

In [ ]:
eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=10,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

TESTING THE DESC TUTORIAL "DESC_TrappedRes/docs/notebooks/tutorials/basic_optimization.ipynb"

In [ ]:
# Copy the basic_optimization.ipynb code here

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    AspectRatio,
    ForceBalance,
    QuasisymmetryBoozer,
    QuasisymmetryTwoTerm,
    QuasisymmetryTripleProduct,
)
from desc.optimize import Optimizer
from desc.plotting import (
    plot_grid,
    plot_boozer_modes,
    plot_boozer_surface,
    plot_qs_error,
    plot_boundaries,
    plot_boundary,
)

# load initial equilibrium
eq_init = desc.io.load("qs_initial_guess.h5")

optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)

eq_qs_T = eq_init.copy()  # make a copy of the original one
# constraints
constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixIota(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# objective
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)
plot_grid(grid_vol, figsize=(8, 8))

objective_fT = ObjectiveFunction(QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol))

eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=50,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

TESTING THE OBJECTIVE FUNCTION VALUE

$ y = \omega_{\zeta} - \omega_{\zeta}^r $

If $|y|<0.5$, this is run:
$
f_{ijr} = A e^{-w ( (y+0.5)^2 + (y+0.5) )^t}
$
Otherwise, 
$
f_{ijr} = 0
$

$ f_{ij} = \sum_{r=1}^{N_r} (f_{ijr}) $, $N_r = $ number of resonances considered.

In [6]:
# With two pitch inverses and three surfaces considered. Use desc_eq_new_QH_aScaling.h5
omega_arr = np.array([[-1.35078567e-04, -2.88730399e-05],[-1.21966454e-05, -2.42665365e-06],[-5.19323354e-06, -1.03486556e-06]]) # omegas outputted from first cell in this document (just runnning the objective function)
res_arr = np.array([0.   ,      1.     ,    0.5    ,    2.]) # resonances outputted from first cell in this document (just runnning the objective function)

In [12]:
import numpy as np
# Calculate objective function (per surface per pitch angle)
res_arr = np.array([0.   ,      1.    ,     0.5    ,    2.    ,     3.14159265])
res_broad = res_arr[None,None,:] # make 3D array with res values on axis=2
res_broad = np.broadcast_to(res_broad, (omega_arr.shape[0], omega_arr.shape[1], res_arr.shape[0]))
omega_broad = np.broadcast_to(omega_arr[...,None], (omega_arr.shape[0],omega_arr.shape[1],res_arr.shape[0]))


y = omega_broad - res_broad
condition = np.logical_and(abs(y) < 0.5, res_broad!=np.pi) # check that corresponding omega value is less than 0.5 away from the resonance and not jnp.pi (unset)
# Set weights
w = 1
t = -1
A = np.ones((np.shape(omega_broad))) * 100 # can set to vary with order of resonance if desired
obj_out = np.where(
    condition,
    A * np.exp(-w * (( -((y+0.5)**2) + (y+0.5) )**t)),
    0
    ) # need to broadcast res_arr to 3D to match each res with each 2D matrix of omega_arr and then do this subtraction and jnp.where operation

obj_out = np.sum(obj_out,axis=2) # outputs array with size (rho,pitch)
print(omega_broad[0,0,:])


/var/folders/1s/jh18wdb5059c0177d80bb_xc0000gn/T/ipykernel_6891/2091222955.py:17: RuntimeWarning: overflow encountered in exp
  A * np.exp(-w * (( -((y+0.5)**2) + (y+0.5) )**t)),


[-0.00013508 -0.00013508 -0.00013508 -0.00013508 -0.00013508]
